# Gold: Product dimension
**Sources:** `silver.crm_products`, `silver.erp_product_category`  →  **Target:** `gold.dim_products`

**What this notebook does:**
- Join products with their category
- Keep **only the current version** of each product (no end date)
- Add a **surrogate key** `product_key`

In [0]:
CATALOG = "workspace"

## The business logic (SQL)
The `WHERE end_date IS NULL` line is important. Without it, old product versions stay in and the sales table gets duplicate rows.

In [0]:
query = f"""
SELECT
    ROW_NUMBER() OVER (ORDER BY pn.start_date, pn.product_number) AS product_key,  -- surrogate key
    pn.product_id,
    pn.product_number,
    pn.product_name,
    pn.category_id,
    pc.category,
    pc.subcategory,
    pc.maintenance_flag,
    pn.product_cost AS cost,
    pn.product_line,
    pn.start_date
FROM {CATALOG}.silver.crm_products pn
LEFT JOIN {CATALOG}.silver.erp_product_category pc
       ON pn.category_id = pc.category_id
WHERE pn.end_date IS NULL          -- current version only: 1 row per product
"""
df = spark.sql(query)

## Preview

In [0]:
df.display()

## Write the Gold table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.dim_products")

## Check it Quickly

In [0]:
result = spark.table(f"{CATALOG}.gold.dim_products")
print("rows:", result.count())